In [16]:
from surprise import Dataset, Reader, KNNBasic, KNNWithMeans, KNNWithZScore, SVD, SVDpp, NMF, CoClustering, accuracy
from surprise.model_selection import cross_validate, GridSearchCV
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from IPython.display import clear_output
from sentence_transformers import SentenceTransformer

import os
import random
import zipfile
import numpy as np      
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
movies = pd.read_csv('../abrikos/cleaned_data/movies_clean.csv', low_memory=False)
ratings = pd.read_csv('../abrikos/cleaned_data/ratings_clean.csv', usecols=['userId', 'movieId', 'rating'], nrows=1_000_000)

In [4]:
movies.shape, ratings.shape

((45433, 25), (1000000, 3))

In [5]:
movies[movies['imdb_id'] == 'tt0114709']
links = pd.read_csv('../abrikos/data_Movies/links.csv')
movies1 = movies.copy()
movies1['imdb_id'] = movies1['imdb_id'].apply(lambda x: str(x).lstrip('tt0'))
movies1['imdb_id'] = pd.to_numeric(movies1['imdb_id'], errors='coerce')
df1 = movies1.merge(links, left_on='imdb_id', right_on='imdbId')
df1.shape

(45353, 28)

In [6]:
df1 = df1.drop_duplicates(subset=['title'])
df1.shape

(42215, 28)

In [7]:
# keep only df1 rows that have movieId in ratings
df2 = df1[df1['movieId'].isin(ratings['movieId'])]
# keep only ratings rows that have movieId in df1
ratings = ratings[ratings['movieId'].isin(df2['movieId'])]

In [8]:
df2.shape, ratings.shape

((18960, 28), (972128, 3))

In [ ]:
train_ratings, test_ratings = train_test_split(ratings, test_size=0.2, shuffle=True, random_state=37)

reader = Reader(rating_scale=(1, 5))

train_data = Dataset.load_from_df(train_ratings, reader)
test_data = Dataset.load_from_df(test_ratings, reader)

In [9]:
# params = {
#     'n_factors': [25, 50, 100],
#     'n_epochs': [10, 20, 30],
#     'lr_all': [0.002, 0.005, 0.01],
#     'reg_all': [0.002, 0.01, 0.05],
# }

# grid_search = GridSearchCV(SVD, param_grid=params, measures=['mae'])
# grid_search.fit(train_data)
# best_model = grid_search.best_estimator['mae']
# clear_output()

In [10]:
# print("Best params by GridSearch:")
# print(grid_search.best_params['mae'])

In [10]:
best_model = SVD(n_epochs=30, n_factors=100,lr_all=0.01,reg_all=0.05)
#best_model.fit(train_data.build_full_trainset())
clear_output()

In [23]:
# 1. Safely extract raw data rows from your DatasetAutoFolds object
# This pulls the raw (userId, movieId, rating) data exactly how it was loaded
raw_interactions = test_data.raw_ratings

# 2. Reformat the raw rows into an explicit list of tuples to pass to the model
testset_tuples = [(uid, iid, r) for (uid, iid, r, _) in raw_interactions]

# 3. Compute model predictions
print("Computing SVD predictions...")
predictions = best_model.test(testset_tuples)

# 4. Let surprise evaluate itself natively
print("\n--- SVD Native Metrics ---")
from surprise import accuracy
mse_score = accuracy.mse(predictions)   
rmse_score = accuracy.rmse(predictions) 

Computing SVD predictions...

--- SVD Native Metrics ---
MSE: 0.3525
RMSE: 0.5937


In [11]:
def get_ratings(n_reviews=5, print_=False):
    new_user_id = len(ratings)

    def add_new_user(n_reviews: int, print_: bool):
        #movies_with_ratings = movies[movies['id'].isin(ratings['movieId'])].reset_index(drop=True)
        # Sample movies that definitely exist
        sample_movies = df2.sample(n=100)
        random_movies_names = sample_movies['title'].tolist()
        random_movie_ids = sample_movies['id'].tolist()

        new_user_ratings = [
            (new_user_id, random_movie_ids[i], random.randint(1, 5))
            for i in range(n_reviews)
        ]
        if print_:
            for movie_name, rating in zip(random_movies_names, [i[2] for i in new_user_ratings]):
                print(movie_name, rating)
        return new_user_ratings

    def model_add_user(n_reviews=5, print_=False):
        # Добавляем информацию о новом пользователе в датасет, заносим данные в модель
        new_user_ratings_df = pd.DataFrame(add_new_user(n_reviews, print_), columns=['userId', 'movieId', 'rating'])
        updated_ratings = pd.concat([ratings, new_user_ratings_df], ignore_index=True)

        updated_data = Dataset.load_from_df(updated_ratings, reader)

        updated_trainset = updated_data.build_full_trainset()

        best_model.fit(updated_trainset)
        return updated_ratings

    updated_ratings = model_add_user(n_reviews, print_)
    # Получаем все записи из датасета рейтингов, кроме записей самого пользователя
    rated_items = set(updated_ratings[updated_ratings['userId'] == new_user_id]['movieId'])
    unrated_items = set(updated_ratings['movieId'].unique()) - rated_items

    # Предсказываем рейтинг для этих записей с ТРЁХ моделей
    predictions = []
    for item_id in list(unrated_items):  # ограничиваем для скорости
        pred_svd = best_model.predict(new_user_id, item_id).est

        predictions.append((item_id, pred_svd))

    # Создаем DataFrame с результатами
    results_df = pd.DataFrame(predictions,
                            columns=['movieId', 'svd'])
    results_df = results_df.sort_values('svd', ascending=False)

    # Топ 5 рекомендаций
    top_n = 10
    top_recs_df = results_df.head(top_n).merge(
        df2[['movieId', 'title']], on='movieId', how='left'
    )[['title', 'svd']]

    # Filter to only movies that exist in movies DataFrame
    #valid_movie_ids = set(df2['m']) & set(results_df['movieId'])
    #results_df_valid = results_df[results_df['movieId'].isin(valid_movie_ids)]

    # top_recs_df = results_df.head(10).merge(
    #     df2, on='movieId', how='left'
    # )[['title', 'svd']]

    print(top_recs_df.round(2).to_string(index=False))

In [ ]:
for n_reviews in [1,5,10,25,50,100]:
    print(f'n_reviews:{n_reviews}')
    get_ratings(n_reviews)
    print()

n_reviews:1
                   title  svd
        Band of Brothers 4.36
            Planet Earth 4.18
            Day of Wrath 4.17
 The Sorrow and the Pity 4.17
The Shawshank Redemption 4.15
                Whiplash 4.15
       Samurai Rebellion 4.12
              The Matrix 4.12
  Ghost in the Shell 2.0 4.10
        Sigur Rós: Heima 4.10

n_reviews:5
                    title  svd
             Planet Earth 3.98
     There Once Was a Dog 3.90
  The Sorrow and the Pity 3.84
  The Power of Nightmares 3.81
  Something the Lord Made 3.81
                Two Women 3.80
        Samurai Rebellion 3.80
              The Leopard 3.79
         Land and Freedom 3.79
The Ballad of Cable Hogue 3.78

n_reviews:10
                                            title  svd
                                  The Celebration 4.27
                            A Man for All Seasons 4.23
Manufacturing Consent: Noam Chomsky and the Media 4.14
                                     Planet Earth 4.13
               

In [12]:
get_ratings(10)

                                            title  svd
                          The Sorrow and the Pity 4.72
                                     Planet Earth 4.66
The Lord of the Rings: The Fellowship of the Ring 4.63
                             Castaway on the Moon 4.57
                             My Favorite Brunette 4.56
    The Lord of the Rings: The Return of the King 4.55
                         The Shawshank Redemption 4.52
                               Sansho the Bailiff 4.52
                                    Frozen Planet 4.50
                             Dylan Moran: Monster 4.49


  title  svd
        Band of Brothers 4.52
 The Sorrow and the Pity 4.52
        Land and Freedom 4.45
            Planet Earth 4.45
    Castaway on the Moon 4.40
 The Power of Nightmares 4.39
              Duck Amuck 4.37
        Rivers and Tides 4.37
           The Civil War 4.36
The Shawshank Redemption 4.36

In [14]:
import pickle
with open('collab_model_svd3', 'wb') as f:
    pickle.dump(best_model, f)

In [15]:
import pickle
with open('collab_model_svd3', 'rb') as f:
     best_model = pickle.load(f)

## Дальше идет Content-based

In [4]:
import faiss

In [5]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8605.84it/s]


In [13]:
movies.columns

Index(['id', 'imdb_id', 'title', 'original_title', 'original_language',
       'overview', 'tagline', 'status', 'adult', 'video', 'budget', 'revenue',
       'runtime', 'vote_count', 'vote_average', 'popularity', 'release_date',
       'release_year', 'genre_names', 'production_company_names',
       'production_country_codes', 'spoken_language_codes', 'poster_path',
       'homepage', 'belongs_to_collection'],
      dtype='object')

In [14]:
movies['title'] = movies['title'].astype(str)
movies = movies.drop_duplicates(subset=['title'])

In [15]:

title_embeddings = model.encode(movies['title'].str.lower().tolist())

In [16]:
titles = movies['title'].str.lower()
genres = movies['genre_names'].str.lower().fillna('')
full_texts = (titles + ' ' + genres).tolist()
text_embeddings = model.encode(full_texts)

# Нормализуем год, если есть (адаптируем, если колонка есть)
if 'year' in movies.columns:  # Извлекаем год из title если нет отдельной колонки
    movies['year'] = movies['title'].str.extract(r'\((\d{4})\)').astype(float)
else:
    movies['year'] = movies['title'].str.extract(r'\((\d{4})\)').astype(float)
scaler = MinMaxScaler()
year_normalized = scaler.fit_transform(movies[['year']].fillna(1900)).flatten().reshape(-1, 1)

# Объединяем эмбеддинги + год
combined_features = np.hstack((text_embeddings, year_normalized))

In [17]:
# Нормализация векторов
norms = np.linalg.norm(combined_features, axis=1)

norms[norms == 0] = 1e-10

normalized_features = combined_features / norms[:, np.newaxis]

In [18]:
# Для эффективной работы с векторами используем faiss
features_float32 = normalized_features.astype('float32')

faiss_index = faiss.IndexFlatIP(features_float32.shape[1])
faiss_index.add(features_float32)

In [19]:
def get_recommendations(user_liked_books, n=5):
    indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()
    liked_indices = [indices[title] for title in user_liked_books]

    liked_vectors = combined_features[np.array(liked_indices)].astype('float32')
    if liked_vectors.ndim == 1:
        liked_vectors = liked_vectors.reshape(1, -1)

    user_vector = np.mean(liked_vectors, axis=0)

    if len(user_vector) != faiss_index.d:
        user_vector = user_vector[:faiss_index.d]

    user_norm = np.linalg.norm(user_vector)
    user_vector_normalized = user_vector / user_norm if user_norm > 0 else user_vector
    user_vector_faiss = user_vector_normalized.reshape(1, -1).astype('float32')

    # Ищем фильмы + получаем скоры (D = косинусные схождения)
    D, I = faiss_index.search(user_vector_faiss, n + len(liked_indices))

    recommended_indices = [idx for idx in I[0] if idx not in liked_indices]
    top_indices = recommended_indices[:n]

    # Нормируем скоры FAISS в 1–5 (чем выше, тем сильнее совпадение)
    top_scores = D[0][:len(top_indices)]
    # Простой линейный рескейл: [min, max] → [1, 5]
    if top_scores.max() != top_scores.min():
        rating = 1 + 4 * (top_scores - top_scores.min()) / (top_scores.max() - top_scores.min())
    else:
        rating = np.full_like(top_scores, 3.0)  # все равны — ставим 3

    result = pd.DataFrame(
        {"title": movies.iloc[top_indices]['title'].tolist(), "rating": rating}
    )
    return result

# Пример использования
random_movies_names = ["The Matrix", "Inception"]  # на твоих данных
recs = get_recommendations(random_movies_names, n=5)
print(recs)

          title    rating
0       Joyride  5.000000
1      Joy Ride  5.000000
2  Sleepwalkers  4.072006
3  The Forsaken  3.267527
4        Riders  1.000000


In [24]:
random_books_names_filtered = ["Toy Story"]

similar_books = get_recommendations(random_books_names_filtered)
print(similar_books)

                  title    rating
0           Toy Story 3  5.000000
1           Toy Story 2  3.701888
2               The Toy  3.682965
3  Toy Story of Terror!  1.463837
4        The Lego Movie  1.000000
